In [64]:
import pandas as pd
import numpy as np

Load the initial data set and do initial data set cleanup by removing unused columns that aren't relevant.

In [65]:
df = pd.read_csv("data/beatmaps.csv")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.8f}'.format)
print(df.columns.to_list())
df = df.drop(columns=['file_md5', 'tags', 'download_unavailable', 'audio_unavailable'])
df.describe()

['beatmapset_id', 'beatmap_id', 'approved', 'total_length', 'hit_length', 'version', 'file_md5', 'diff_size', 'diff_overall', 'diff_approach', 'diff_drain', 'mode', 'count_normal', 'count_slider', 'count_spinner', 'submit_date', 'approved_date', 'last_update', 'artist', 'title', 'creator', 'creator_id', 'bpm', 'source', 'tags', 'genre_id', 'language_id', 'favourite_count', 'rating', 'download_unavailable', 'audio_unavailable', 'playcount', 'passcount', 'max_combo', 'diff_aim', 'diff_speed', 'difficultyrating']


,beatmapset_id,beatmap_id,approved,total_length,hit_length,diff_size,diff_overall,diff_approach,diff_drain,mode,count_normal,count_slider,count_spinner,creator_id,bpm,genre_id,language_id,favourite_count,rating,playcount,passcount,max_combo,diff_aim,diff_speed,difficultyrating
count,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,83640.00000000,67868.00000000,67868.00000000,65221.00000000,83640.00000000
mean,335306.94646102,756453.02956719,1.16576997,144.42599235,132.97595648,3.93959908,5.73278659,6.36860844,5.10115674,0.40784314,326.48511478,134.94356767,2.05057389,2144343.10487805,71902.93435323,4.02846724,3.35346724,253.23602343,9.12377096,127889.25154232,30322.46385701,571.23442565,1.78091389,1.53467364,3.36456934
std,307247.44094337,643218.54490012,0.67615548,79.18923352,72.16143526,0.97796036,2.12649160,2.25347491,1.79207512,0.87907300,519.49218863,151.88355728,11.39051170,2340454.08591742,20746470.66464403,2.60620423,1.33256436,499.42395253,0.45702579,371057.40940144,94382.06214076,448.66936104,2.94197302,0.65584022,4.06946686
min,1.00000000,53.00000000,-2.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,2.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
25%,41216.00000000,135660.25000000,1.00000000,89.00000000,87.00000000,3.20000000,4.00000000,5.00000000,4.00000000,0.00000000,83.00000000,51.00000000,0.00000000,207340.00000000,133.00000000,2.00000000,3.00000000,51.00000000,8.98310500,16280.75000000,4212.00000000,301.00000000,1.06338933,0.95710236,2.08163357
50%,257565.00000000,647446.50000000,1.00000000,118.00000000,111.00000000,4.00000000,6.00000000,7.00000000,5.00000000,0.00000000,175.00000000,103.00000000,1.00000000,1352672.00000000,160.00000000,3.00000000,3.00000000,116.00000000,9.21409000,43252.00000000,11307.00000000,458.00000000,1.62546146,1.49636018,3.18924546
75%,596902.00000000,1324399.50000000,1.00000000,192.00000000,168.00000000,4.00000000,7.50000000,8.00000000,6.00000000,0.00000000,362.00000000,171.00000000,3.00000000,3344567.00000000,180.00000000,5.00000000,5.00000000,261.00000000,9.37500000,115568.75000000,29074.25000000,681.00000000,2.26290464,2.01917768,4.33394754
max,1007573.00000000,2113339.00000000,4.00000000,4336.00000000,4176.00000000,10.00000000,10.00000000,10.00000000,10.00000000,3.00000000,38130.00000000,6202.00000000,2872.00000000,13107354.00000000,6000000000.00000000,10.00000000,11.00000000,15419.00000000,10.00000000,25960995.00000000,9724808.00000000,10540.00000000,726.73339844,9.94385147,1091.74304199


Adjust the data by removing rows that are not relevant to this project. Specifically,
Remove all rows where mode is not 0 (the standard osu game mode).
Remove all rows that are either pending (0), WIP (-1), or graveyard (-2)
Remove all rows where the total song time is less than 40 seconds or greater than 2 standard deviations from the mean.
Remove all rows where the Drain Ratio (Hit length / Total Length) is less than 0.6, This means that the map is less than 60% game time and a large part is empty audio (intros, outros, massive breaks) and is a good indicator of a low-quality beatmap.
Any map that falls into one of these categories will be considered as these maps are generally incomplete or of low quality (meme maps, endurance maps, practice maps, TV size, etc)

In [66]:
# Drop the rows using their indices, modifying the DataFrame in-place
df.drop(df[df['mode'] != 0].index, inplace= True)
df.drop(df[df['approved'].isin([0, -1, -2])].index, inplace=True)
#calculate bounds
total_length_mean_val = df['total_length'].mean()
total_length_std_val = df['total_length'].std()
hit_length_mean_val = df['hit_length'].mean()
hit_length_std_val = df['hit_length'].std()
min_total_length = 40
max_total_length = total_length_mean_val + 2 * total_length_std_val
max_hit_length = hit_length_mean_val + 2 * hit_length_std_val
# Overwrite df with only the rows that meet the length criteria
df = df[
    (df['total_length'] >= min_total_length) & 
    (df['total_length'] <= max_total_length) &
    (df['hit_length'] >= min_total_length) &
    (df['hit_length'] <= max_hit_length)
]

# Calculate drain ratio, keep the high-quality maps, and drop the temporary column
df['drain_ratio'] = df['hit_length'] / df['total_length']
df = df[df['drain_ratio'] >= 0.60]
df = df.drop(columns=['drain_ratio'])
df.describe()


,beatmapset_id,beatmap_id,approved,total_length,hit_length,diff_size,diff_overall,diff_approach,diff_drain,mode,count_normal,count_slider,count_spinner,creator_id,bpm,genre_id,language_id,favourite_count,rating,playcount,passcount,max_combo,diff_aim,diff_speed,difficultyrating
count,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000,61814.00000000
mean,300250.09769631,680438.55011810,1.08630731,139.68932604,126.37082214,3.77916082,5.50406801,6.36086696,4.74890268,0.00000000,181.07517714,139.75338920,2.23607921,1910800.14705407,97233.43085854,3.98773741,3.25744977,278.28943281,9.12484244,152443.83227101,35794.52130585,526.94525512,1.67410629,1.50965810,3.28218671
std,307703.37069985,646074.21772661,0.48577637,59.34265503,51.16822936,0.67281463,2.18215114,2.28793393,1.74680562,0.00000000,156.70398804,95.61847047,2.61710473,2308149.88739252,24132805.59414478,2.47203538,1.30004340,537.58852523,0.43457134,413325.30515555,106683.51748360,317.09264075,3.00530862,0.63657641,4.59745302
min,1.00000000,53.00000000,1.00000000,40.00000000,40.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,2.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,33.00000000,0.23990998,0.13304779,0.42951739
25%,27378.00000000,95643.50000000,1.00000000,89.00000000,87.00000000,3.00000000,4.00000000,5.00000000,3.00000000,0.00000000,71.00000000,73.00000000,1.00000000,131270.00000000,131.00000000,2.00000000,3.00000000,60.00000000,8.98753750,24089.00000000,6534.00000000,303.00000000,1.04109055,0.95027240,2.04604757
50%,164782.50000000,433346.50000000,1.00000000,119.00000000,109.00000000,4.00000000,6.00000000,7.00000000,5.00000000,0.00000000,136.00000000,117.00000000,2.00000000,896613.00000000,160.00000000,3.00000000,3.00000000,131.00000000,9.22057000,56829.00000000,14675.00000000,452.00000000,1.56263542,1.47024435,3.11658752
75%,550486.00000000,1224676.75000000,1.00000000,189.00000000,164.00000000,4.00000000,7.00000000,8.00000000,6.00000000,0.00000000,242.00000000,176.00000000,3.00000000,3044645.00000000,180.00000000,5.00000000,3.00000000,284.00000000,9.37241000,142117.75000000,34633.75000000,649.00000000,2.15218210,1.98149833,4.25100744
max,1007573.00000000,2113339.00000000,4.00000000,299.00000000,268.00000000,8.00000000,10.00000000,10.00000000,10.00000000,0.00000000,1955.00000000,1106.00000000,212.00000000,13107354.00000000,6000000000.00000000,10.00000000,11.00000000,15419.00000000,10.00000000,25960995.00000000,9724808.00000000,4327.00000000,726.73339844,7.41724682,1091.74304199
